# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the open FAIR² Mass Spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) Python library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all available Record Sets, their `@id` fields, and each Record Set's Fields and Columns as defined in the Croissant schema.

In [ ]:
# List all available record sets with their @id and field @ids
from mlcroissant.types import RecordSet
if not hasattr(metadata_obj, 'record_sets'):
    # For mlcroissant < 0.5.0
    record_sets = metadata_obj.record_set if hasattr(metadata_obj, 'record_set') else []
else:
    record_sets = metadata_obj.record_sets

print("Record Sets found in dataset:\n")
record_set_ids = []
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {rs.id} | name: {getattr(rs, 'name', '<no name>')}")
    field_ids = [field.id for field in getattr(rs, 'fields', [])]
    print(f"    Fields (@id): {field_ids}")
    column_ids = []
    for field in getattr(rs, 'fields', []):
        if hasattr(field, 'column'):
            if isinstance(field.column, list):
                column_ids += [c.id for c in field.column]
            else:
                column_ids.append(field.column.id)
    print(f"    Columns (@id): {column_ids}\n")
    record_set_ids.append(rs.id)
if len(record_sets) == 0:
    print('No record sets found in metadata. Please check the schema definition.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

Replace the `<record_set_id>` below with the actual `@id` of the record set you want to analyze (printed above). 
If there is more than one record set, you can loop over all of them as well.

In [ ]:
# Extract data from each record set using their @id
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

dataframes = {}

for record_set_id in record_set_ids:
    try:
        print(f"Loading records for RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame with columns: {list(df.columns)}")
        print(df.head(2))
    except Exception as e:
        print(f"Failed to load {record_set_id}: {e}\n")

# Pick the first (main) record set if found
if record_set_ids:
    records_set_id = record_set_ids[0]
    print(f"Main DataFrame columns: {dataframes[records_set_id].columns.tolist()}")
    display(dataframes[records_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You will need to select a numeric field from the DataFrame above by its `@id` (column name).

In [ ]:
# Example field selection for EDA
df = dataframes[records_set_id]
print(f"Available columns: {df.columns.tolist()}")

# Choose a numeric field by @id. Replace with actual id as needed.
possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64','int64']]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # Attempt to infer numeric-looking columns
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
            numeric_field = col
            break
        except:
            continue
    else:
        numeric_field = df.columns[0]  # fallback
print(f"Using {numeric_field} as a numeric field for analysis.")

# Try converting the field if it's not in numeric dtype
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter and normalize
threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available
group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
if group_fields:
    group_field = group_fields[0]
    print(f"Grouping by {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib for histogram and boxplot of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    df[numeric_field].hist(bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')

    plt.subplot(1,2,2)
    df[numeric_field].plot.box()
    plt.title(f"Boxplot of {numeric_field}")
    plt.ylabel(numeric_field)
    plt.show()

if 'grouped_df' in locals():
    plt.figure(figsize=(8,4))
    plt.bar(grouped_df[group_field], grouped_df[numeric_field], color='royalblue')
    plt.xticks(rotation=90)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
We demonstrated how to load and explore the FAIR² Mass Spectrometry dataset using `mlcroissant`.

- **Schema navigation**: We used Record Set and Field `@id`s for dynamic and standards-based data access.
- **Custom analysis**: We filtered, normalized, and grouped numeric data to prepare it for downstream tasks.
- **Visualization**: Histograms and bar plots enable rapid insight into distribution and grouping structures.

Further analysis can reference fields and record sets by their `@id` to maintain traceability and reproducibility.